## 1. Import library dan pengaturan

In [1]:
import hashlib
import random
import shutil
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PRAPROSES = ROOT / "data" / "praproses"
LABEL = ROOT / "data" / "labels"
DATASET = ROOT / "dataset"
HASIL = ROOT / "hasil" / "pembagian_dataset"

KELAS = ["logam", "paper", "plastic"]
SEED = 42
RASIO_VAL, RASIO_TEST = 0.2, 0.1      # sisanya (70%) untuk training

## 2. Kumpulkan foto yang sudah punya label

In [2]:
tabel = pd.read_csv(ROOT / "hasil" / "praproses" / "praproses.csv")
tabel = tabel[tabel["nama_baru"].map(lambda n: (LABEL / n.replace(".jpg", ".txt")).exists())]
if tabel.empty:
    raise FileNotFoundError("Belum ada label. Jalankan 05_anotasi_bbox.ipynb dulu.")
print(f"Foto berlabel: {len(tabel)}")

Foto berlabel: 394


## 3. Bagi dataset per kelas

In [3]:
if DATASET.exists():
    shutil.rmtree(DATASET)          # dataset lama (termasuk hasil augmentasi) dibuat ulang

acak = random.Random(SEED)
pembagian = []
for kelas in KELAS:
    nama = sorted(tabel[tabel["kelas"] == kelas]["nama_baru"])
    acak.shuffle(nama)
    n = len(nama)
    n_test = max(1, round(n * RASIO_TEST)) if n >= 3 else 0
    n_val = max(1, round(n * RASIO_VAL)) if n >= 3 else 0
    n_train = n - n_val - n_test
    bagian = ["train"] * n_train + ["val"] * n_val + ["test"] * n_test
    pembagian += [dict(nama=a, kelas=kelas, bagian=b) for a, b in zip(nama, bagian)]

pembagian = pd.DataFrame(pembagian)

for r in pembagian.itertuples():
    (DATASET / "images" / r.bagian).mkdir(parents=True, exist_ok=True)
    (DATASET / "labels" / r.bagian).mkdir(parents=True, exist_ok=True)
    shutil.copy2(PRAPROSES / r.kelas / r.nama, DATASET / "images" / r.bagian / r.nama)
    shutil.copy2(LABEL / r.nama.replace(".jpg", ".txt"), DATASET / "labels" / r.bagian / r.nama.replace(".jpg", ".txt"))

HASIL.mkdir(parents=True, exist_ok=True)
pembagian.to_csv(HASIL / "pembagian.csv", index=False)

## 4. Jumlah foto per kelas pada tiap bagian

In [4]:
jumlah = pd.crosstab(pembagian["kelas"], pembagian["bagian"])[["train", "val", "test"]]
jumlah.loc["total"] = jumlah.sum()
display(jumlah)

persen = (jumlah.loc["total"] / jumlah.loc["total"].sum() * 100).round(1)
print("Persentase terhadap seluruh data (%):", persen.to_dict())

bagian,train,val,test
kelas,,,
logam,86,25,12
paper,107,30,15
plastic,83,24,12
total,276,79,39


Persentase terhadap seluruh data (%): {'train': 70.1, 'val': 20.1, 'test': 9.9}


## 5. Pemeriksaan kebocoran data


In [5]:
# (a) nama file unik
duplikat_nama = pembagian["nama"][pembagian["nama"].duplicated()].tolist()

# (b) isi file identik di bagian berbeda
sidik, bocor = {}, []
for r in pembagian.itertuples():
    h = hashlib.md5((DATASET / "images" / r.bagian / r.nama).read_bytes()).hexdigest()
    if h in sidik and sidik[h][1] != r.bagian:
        bocor.append((sidik[h][0], r.nama))
    sidik.setdefault(h, (r.nama, r.bagian))

if duplikat_nama or bocor:
    print("PERINGATAN, ada kebocoran data:", duplikat_nama, bocor)
else:
    print("Aman: tidak ada foto yang muncul di lebih dari satu bagian dan tidak ada isi file identik antar bagian.")

Aman: tidak ada foto yang muncul di lebih dari satu bagian dan tidak ada isi file identik antar bagian.
